# Quick MEM — 1D Mechanical Earth Model Workflow

End-to-end demo of **GeomechPy** following the standard MEM sequence:

1. Data input (pandas)
2. Overburden stress
3. Lithology (toolbox)
4. Pore pressure
5. Dynamic elastic properties
6. Static elastic properties
7. Rock strength
8. Horizontal stresses + stress-tensor rotation
9. Wellbore stability (breakout / breakdown)

Synthetic log data is generated so the notebook runs without external files.


## 0. Setup & imports


In [1]:
import sys
from pathlib import Path

# repo root = two levels up from this notebook (example/Project/)
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "Project":
    REPO_ROOT = REPO_ROOT.parent.parent
elif REPO_ROOT.name == "example":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "example"))  # lithology in example/toolbox.py

import numpy as np
import pandas as pd

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
# Interactive plots that render in JupyterLab, VS Code and nbviewer.
pio.renderers.default = "plotly_mimetype+notebook_connected"

from geomechpy.overburden_stress import OverburdenStressCalculation
from geomechpy.pore_pressure import PorePressureCalculation
from geomechpy.elastic_properties import ElasticPropertiesConverter
from geomechpy.static_elastic_properties import StaticElasticPropertiesConverter
from geomechpy.rock_strength import RockStrengthPropertiesConverter
from geomechpy.stress_calculations import HorizontalStressesCalculation
from geomechpy.wellbore_stability import WellboreStabilityCalculation
from geomechpy.toolbox import rotate_stress_to_shmax, rotate_nev_to_toh
from toolbox import determine_lithology_array  # example/toolbox.py

print("Imports OK — repo root:", REPO_ROOT)

Imports OK — repo root: /home/user/GeomechPy_smolrun


## 1. Data input

Synthetic onshore well log (TVD from 5 000 – 8 000 ft).
Columns mimic typical LAS curves used in a 1-D MEM.


In [2]:
np.random.seed(42)
n = 61
tvd = np.linspace(5000.0, 8000.0, n)  # ft

# Synthetic curves with mild depth trends + noise
gr = 40 + 40 * np.sin(np.linspace(0, 4 * np.pi, n)) + np.random.normal(0, 5, n)  # gAPI
gr = np.clip(gr, 15, 140)

rhob = 2.35 + 0.00004 * (tvd - 5000) + np.random.normal(0, 0.02, n)  # g/cm3
dtco = 90 - 0.005 * (tvd - 5000) + np.random.normal(0, 2, n)       # us/ft
dtsh = 160 - 0.008 * (tvd - 5000) + np.random.normal(0, 3, n)      # us/ft
nphi = 0.18 - 0.00002 * (tvd - 5000) + np.random.normal(0, 0.01, n) # fraction
nphi = np.clip(nphi, 0.05, 0.30)

# Optional coal / limestone flags (mostly False)
coal_flag = [False] * n
limestone_flag = [False] * n
coal_flag[20] = True
limestone_flag[45] = True

df = pd.DataFrame({
    "TVD": tvd,
    "GR": gr,
    "RHOB": rhob,
    "DTCO": dtco,
    "DTSH": dtsh,
    "NPHI": nphi,
    "COAL": coal_flag,
    "LIME": limestone_flag,
})
df.head()


,TVD,GR,RHOB,DTCO,DTSH,NPHI,COAL,LIME
0,5000.0,42.483571,2.346287,92.805589,161.447417,0.167997,False,False
1,5050.0,47.625146,2.329873,86.946298,158.929612,0.175655,False,False
2,5100.0,59.507908,2.330076,90.673714,161.342001,0.173251,False,False
3,5150.0,71.126559,2.372251,93.630911,160.219713,0.170467,False,False
4,5200.0,68.555026,2.385125,87.018927,158.181513,0.193655,False,False


## 2. Overburden stress (onshore)


In [3]:
AIR_GAP = 30.0  # ft KB to ground
LITHOSTATIC_GRAD = 1.05  # psi/ft

df["SV"] = OverburdenStressCalculation.calculate_overburden_stress_onshore_array(
    tvd=df["TVD"].tolist(),
    lithostatic_gradient=LITHOSTATIC_GRAD,
    air_gap=AIR_GAP,
)
df[["TVD", "SV"]].head()


,TVD,SV
0,5000.0,5218.512
1,5050.0,5271.012
2,5100.0,5323.512
3,5150.0,5376.012
4,5200.0,5428.512


## 3. Lithology (mechanical stratigraphy from GR)


In [4]:
df["LITH"] = determine_lithology_array(
    gamma_ray=df["GR"].tolist(),
    gr_threshold=75.0,
    coal_flag=df["COAL"].tolist(),
    limestone_flag=df["LIME"].tolist(),
)
# 0=Sand, 1=Shale, 2=Limestone, 6=Coal
df["LITH_NAME"] = df["LITH"].map({0: "Sand", 1: "Shale", 2: "Limestone", 6: "Coal"})
df[["TVD", "GR", "LITH", "LITH_NAME"]].value_counts("LITH_NAME")


LITH_NAME
Sand         52
Shale         7
Coal          1
Limestone     1
Name: count, dtype: int64

## 4. Pore pressure (onshore hydrostatic)


In [5]:
PP_GRAD = 0.465  # psi/ft

df["PP"] = PorePressureCalculation.calculate_pore_pressure_onshore_array(
    tvd=df["TVD"].tolist(),
    formation_pore_pressure_gradient=PP_GRAD,
    air_gap=AIR_GAP,
)
df[["TVD", "PP"]].head()


,TVD,PP
0,5000.0,2311.062
1,5050.0,2334.312
2,5100.0,2357.562
3,5150.0,2380.812
4,5200.0,2404.062


## 5. Dynamic elastic properties

From compressional / shear slowness + bulk density.


In [6]:
# density must be kg/m3 for the converter
density_kgm3 = (df["RHOB"] * 1000).tolist()

dyn_props = ElasticPropertiesConverter.convert_dynamic_elastic_properties_from_slowness_array(
    p_wave_slowness=df["DTCO"].tolist(),
    s_wave_slowness=df["DTSH"].tolist(),
    density=density_kgm3,
)

df["YME_DYN_Pa"] = [p.youngs_modulus for p in dyn_props]
df["PR_DYN"] = [p.poissons_ratio for p in dyn_props]
df["G_DYN_Pa"] = [p.shear_modulus for p in dyn_props]

# Convert Pa to Mpsi for static correlations that expect Mpsi
PA_TO_MPSI = 1.0 / 6.894757e9
df["YME_DYN_Mpsi"] = df["YME_DYN_Pa"] * PA_TO_MPSI
df[["TVD", "YME_DYN_Mpsi", "PR_DYN"]].head()


,TVD,YME_DYN_Mpsi,PR_DYN
0,5000.0,3.040159,0.253246
1,5050.0,3.197808,0.286438
2,5100.0,3.061527,0.269176
3,5150.0,3.089803,0.240684
4,5200.0,3.295894,0.283018


## 6. Static elastic properties

Bradford correlation (dynamic to static YME) + simple PR scaling.


In [7]:
df["YME_STA_Mpsi"] = StaticElasticPropertiesConverter.dyn2sta_yme_bradord_array(
    yme_dyn=df["YME_DYN_Mpsi"].tolist()
)
df["PR_STA"] = StaticElasticPropertiesConverter.dyn2sta_poissons_ratio_array(
    pr_dyn=df["PR_DYN"].tolist(),
    multiplier=1.0,
)
df["BIOT"] = [
    StaticElasticPropertiesConverter.biot_coefficient_constant_law(1.0)
    for _ in range(len(df))
]
df[["TVD", "YME_STA_Mpsi", "PR_STA", "BIOT"]].head()


,TVD,YME_STA_Mpsi,PR_STA,BIOT
0,5000.0,0.965106,0.253246,1.0
1,5050.0,1.106258,0.286438,1.0
2,5100.0,0.983530,0.269176,1.0
3,5150.0,1.008250,0.240684,1.0
4,5200.0,1.200280,0.283018,1.0


## 7. Rock strength (UCS, tensile strength, friction angle)


In [8]:
df["UCS"] = RockStrengthPropertiesConverter.convert_yme_sta_to_ucs_plumb_array(
    yme_sta=df["YME_STA_Mpsi"].tolist()
)
df["TSTR"] = RockStrengthPropertiesConverter.convert_ucs_to_tstr_array(
    ucs=df["UCS"].tolist(),
    multiplier=0.15,
)
df["FANG"] = RockStrengthPropertiesConverter.convert_friction_angle_lal_array(
    dtco=df["DTCO"].tolist()
)
df[["TVD", "UCS", "TSTR", "FANG"]].head()


,TVD,UCS,TSTR,FANG
0,5000.0,0.202968,0.030445,32.220415
1,5050.0,0.232653,0.034898,33.787152
2,5100.0,0.206843,0.031026,32.781899
3,5150.0,0.212042,0.031806,32.005581
4,5200.0,0.252427,0.037864,33.767269


## 8. Horizontal stresses (poroelastic) + stress-tensor rotation

Poroelastic model gives Shmin / Shmax, then rotate principal stresses into NEV frame.


In [9]:
hs_list = HorizontalStressesCalculation.calculate_poroelastic_horizontal_stresses_array(
    overburden_stress=df["SV"].tolist(),
    pore_pressure=df["PP"].tolist(),
    poisson_ratio=df["PR_STA"].tolist(),
    youngs_modulus=df["YME_STA_Mpsi"].tolist(),
    biot_coefficient=df["BIOT"].tolist(),
    EX=0.0001,
    EY=0.0005,
)
df["SHMIN"] = [h.shmin for h in hs_list]
df["SHMAX"] = [h.shmax for h in hs_list]
df["Q_FACTOR"] = [h.q_factor for h in hs_list]
df["SHMAX_SHMIN_RATIO"] = [h.shmax_shmin_ratio for h in hs_list]

# Example rotation at mid-depth
mid = len(df) // 2
SHMAX_AZIMUTH = 45.0  # deg from North
stress_nev = rotate_stress_to_shmax(
    shmin=df["SHMIN"].iloc[mid],
    shmax=df["SHMAX"].iloc[mid],
    svert=df["SV"].iloc[mid],
    shmax_azimuth=SHMAX_AZIMUTH,
)
print("Stress tensor NEV at mid-depth (psi):")
print(np.round(stress_nev, 1))
df[["TVD", "SHMIN", "SHMAX", "Q_FACTOR"]].head()


Stress tensor NEV at mid-depth (psi):
[[4.5779e+03 3.0000e-01 0.0000e+00]
 [3.0000e-01 4.5779e+03 0.0000e+00]
 [0.0000e+00 0.0000e+00 6.7935e+03]]


,TVD,SHMIN,SHMAX,Q_FACTOR
0,5000.0,3297.299162,3297.607196,2.999907
1,5050.0,3513.453854,3513.797830,2.999902
2,5100.0,3450.226000,3450.535974,2.999910
3,5150.0,3330.451351,3330.776414,2.999902
4,5200.0,3598.231561,3598.605766,2.999896


## 9. Wellbore stability (vertical well)

Breakout (Mohr-Coulomb) and breakdown (tensile) pressures.


In [10]:
df["PW_BREAKOUT"] = (
    WellboreStabilityCalculation
    .calculate_breakout_calculation_vertical_well_mohr_coulomb_analytical_array(
        shmax=df["SHMAX"].tolist(),
        shmin=df["SHMIN"].tolist(),
        pprs=df["PP"].tolist(),
        overburden_stress=df["SV"].tolist(),
        ucs=df["UCS"].tolist(),
        fang=df["FANG"].tolist(),
        pr_sta=df["PR_STA"].tolist(),
    )
)
df["PW_BREAKDOWN"] = (
    WellboreStabilityCalculation
    .calculate_breakdown_calculation_vertical_well_analytical_array(
        shmax=df["SHMAX"].tolist(),
        shmin=df["SHMIN"].tolist(),
        pprs=df["PP"].tolist(),
        tstr=df["TSTR"].tolist(),
    )
)
df[["TVD", "PW_BREAKOUT", "PW_BREAKDOWN", "PP"]].head()


,TVD,PW_BREAKOUT,PW_BREAKDOWN,PP
0,5000.0,3196.308643,4283.258735,2311.062
1,5050.0,3172.015486,4692.286631,2334.312
2,5100.0,3239.878281,4542.611052,2357.562
3,5150.0,3300.884263,4279.797447,2380.812
4,5200.0,3267.516013,4792.064781,2404.062


## Summary plot — key MEM curves


In [11]:
# Interactive 6-track composite log (hover, zoom, pan; depth increases downward)
fig = make_subplots(
    rows=1, cols=6, shared_yaxes=True, horizontal_spacing=0.015,
    subplot_titles=("GR (gAPI)", "Stresses (ksi)", "YME_sta (Mpsi)",
                    "PR_sta", "UCS (psi)", "Mud window (ksi)"),
)

# Track 1 - Gamma Ray
fig.add_trace(go.Scatter(x=df["GR"], y=df["TVD"], mode="lines",
                         line=dict(color="green"), name="GR"), row=1, col=1)

# Track 2 - stress profiles
for col, color, name in [("SV", "black", "Sv"), ("SHMAX", "red", "SHmax"),
                         ("SHMIN", "blue", "Shmin"), ("PP", "cyan", "Pp")]:
    fig.add_trace(go.Scatter(x=df[col] / 1000, y=df["TVD"], mode="lines",
                             line=dict(color=color), name=name), row=1, col=2)

# Track 3 - static Young's modulus
fig.add_trace(go.Scatter(x=df["YME_STA_Mpsi"], y=df["TVD"], mode="lines",
                         line=dict(color="magenta"), name="YME_sta",
                         showlegend=False), row=1, col=3)

# Track 4 - static Poisson's ratio
fig.add_trace(go.Scatter(x=df["PR_STA"], y=df["TVD"], mode="lines",
                         line=dict(color="orange"), name="PR_sta",
                         showlegend=False), row=1, col=4)

# Track 5 - UCS (psi; note the geomechpy Plumb correlation returns small values)
fig.add_trace(go.Scatter(x=df["UCS"], y=df["TVD"], mode="lines",
                         line=dict(color="brown"), name="UCS",
                         showlegend=False), row=1, col=5)

# Track 6 - mud window
for col, color, dash, name, show in [
    ("PW_BREAKOUT", "red", None, "Breakout", True),
    ("PW_BREAKDOWN", "blue", None, "Breakdown", True),
    ("PP", "cyan", "dash", "Pp", False),
]:
    fig.add_trace(go.Scatter(x=df[col] / 1000, y=df["TVD"], mode="lines",
                             line=dict(color=color, dash=dash), name=name,
                             showlegend=show), row=1, col=6)

fig.update_yaxes(autorange="reversed", title_text="TVD (ft)", row=1, col=1)
fig.update_layout(
    height=800, width=1150, template="plotly_white",
    title="Quick MEM — Synthetic Well",
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="left", x=0),
    margin=dict(t=110),
)
fig.show()

## Done

The DataFrame `df` now holds a complete 1-D MEM at every depth sample.
Export with:

```python
df.to_csv("quick_mem_results.csv", index=False)
```
